# LeetCode #143 — Reorder List: tìm giữa + đảo nửa sau + trộn xen kẽ

**Mức độ:** Medium · **Chủ đề:** Linked List, Two Pointers, Stack, Recursion

Skeleton trên LeetCode:

```python
class Solution:
    def reorderList(self, head: Optional[ListNode]) -> None:
        """
        Do not return anything, modify head in-place instead.
        """
```

Bài này ghép **ba kỹ thuật cơ bản** của danh sách liên kết thành một:

1. **Tìm giữa** bằng con trỏ nhanh/chậm (#876).
2. **Đảo ngược** nửa sau (#206).
3. **Trộn xen kẽ** hai danh sách (giống bước trộn của #21).

Ngoài ra: cách dùng mảng/stack (`O(n)` bộ nhớ) để có lời giải dễ kiểm chứng.

# 1. Bài toán là gì?

Danh sách `L0 → L1 → … → Ln−1 → Ln`. Sắp xếp lại **tại chỗ** thành:

```text
L0 → Ln → L1 → Ln−1 → L2 → Ln−2 → …
```

Chỉ được thay đổi **các nút** (con trỏ), không được sửa **giá trị**.

**Constraints (theo đề):**

- Số nút trong khoảng `[1, 5 * 10^4]`.
- `1 <= Node.val <= 1000`

---

## Ví dụ

```text
1 → 2 → 3 → 4         ->  1 → 4 → 2 → 3
1 → 2 → 3 → 4 → 5     ->  1 → 5 → 2 → 4 → 3
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Danh sách liên kết đơn (singly linked list)

Cuối mục có phần **2.2 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Danh sách liên kết đơn (singly linked list)

### Danh sách liên kết là gì?

Mảng (`list` của Python) lưu các phần tử **liền nhau** trong bộ nhớ, nên truy cập `a[i]` tức thì. Danh sách liên kết thì khác: mỗi phần tử là một **nút** (node) riêng lẻ, chứa **giá trị** và **con trỏ** `next` chỉ tới nút kế tiếp. Nút cuối có `next = None`.

```text
head
 │
 ▼
┌───┬───┐   ┌───┬───┐   ┌───┬────┐
│ 1 │ ●─┼──►│ 2 │ ●─┼──►│ 3 │None│
└───┴───┘   └───┴───┘   └───┴────┘
 val next
```

Ta chỉ giữ con trỏ tới nút đầu (`head`). Muốn tới nút thứ `k` phải đi qua `k` bước — **không** có truy cập ngẫu nhiên.

| | Mảng | Danh sách liên kết |
|---|---|---|
| Lấy phần tử thứ `i` | `O(1)` | `O(i)` |
| Chèn/xóa ở đầu | `O(n)` | `O(1)` |
| Chèn/xóa sau một nút đã có trong tay | `O(n)` | `O(1)` |

In [1]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

# Tạo bằng tay: 1 -> 2 -> 3
c = ListNode(3)
b = ListNode(2, c)
a = ListNode(1, b)
head = a
print(head.val, head.next.val, head.next.next.val, head.next.next.next)

1 2 3 None


### Duyệt danh sách và hai hàm tiện ích

Mẫu duyệt cơ bản: con trỏ `cur` bắt đầu ở `head`, mỗi bước `cur = cur.next`, dừng khi `cur is None`.

In [2]:
def tao_danh_sach(values):
    """[1, 2, 3] -> 1 -> 2 -> 3, trả về head."""
    dummy = tail = ListNode()          # nút giả để khỏi xử lý riêng nút đầu
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next

def ra_list(head):
    out, cur = [], head
    while cur:                          # duyệt tới khi hết
        out.append(cur.val)
        cur = cur.next
    return out

h = tao_danh_sach([4, 5, 6, 7])
print("ra_list:", ra_list(h))
dem, cur = 0, h
while cur:
    dem += 1
    cur = cur.next
print("độ dài:", dem)

ra_list: [4, 5, 6, 7]
độ dài: 4


### Chèn và xóa: chỉ cần đổi con trỏ

Chèn nút `x` **sau** nút `p`: `x.next = p.next; p.next = x` (thứ tự quan trọng — làm ngược sẽ mất phần sau).

Xóa nút **sau** `p`: `p.next = p.next.next`.

```text
Chèn x sau p:   p ──► q        ⟹   p ──► x ──► q
Xóa sau p:      p ──► q ──► r  ⟹   p ──────────► r
```

Muốn xóa một nút, ta cần nút **đứng trước** nó. Nếu nút cần xóa là `head` thì không có nút trước → dùng **nút giả (dummy)** đặt trước `head`.

In [3]:
h = tao_danh_sach([1, 2, 4])
p = h.next                              # nút 2
x = ListNode(3)
x.next = p.next                         # 3 -> 4
p.next = x                              # 2 -> 3
print("sau khi chèn 3:", ra_list(h))

def xoa_gia_tri(head, target):
    dummy = ListNode(0, head)           # dummy đứng trước head
    prev = dummy
    while prev.next:
        if prev.next.val == target:
            prev.next = prev.next.next  # bỏ qua nút cần xóa
        else:
            prev = prev.next
    return dummy.next                   # head mới (có thể đã đổi)

print("xóa mọi số 1:", ra_list(xoa_gia_tri(tao_danh_sach([1, 1, 2, 1, 3]), 1)))

sau khi chèn 3: [1, 2, 3, 4]
xóa mọi số 1: [2, 3]


### Đảo ngược danh sách — kỹ thuật ba con trỏ

Đi dọc danh sách, lật mũi tên của từng nút cho trỏ về nút **trước** nó. Cần lưu `nxt` trước khi lật, vì sau khi lật ta mất đường đi tiếp.

```text
prev  cur   nxt
None   1 ──► 2 ──► 3          bước 1: 1.next = None
None ◄─1     2 ──► 3          bước 2: 2.next = 1
None ◄─1 ◄─ 2     3          bước 3: 3.next = 2  → head mới = 3
```

In [4]:
def dao_nguoc(head):
    prev, cur = None, head
    while cur:
        nxt = cur.next      # 1) lưu đường đi tiếp
        cur.next = prev     # 2) lật mũi tên
        prev = cur          # 3) tiến prev
        cur = nxt           # 4) tiến cur
    return prev             # prev là head mới

print("đảo [1..5]:", ra_list(dao_nguoc(tao_danh_sach([1, 2, 3, 4, 5]))))

đảo [1..5]: [5, 4, 3, 2, 1]


### Con trỏ nhanh / chậm (fast & slow)

`slow` đi 1 bước, `fast` đi 2 bước mỗi vòng.

- Khi `fast` tới cuối, `slow` ở **giữa** danh sách.
- Nếu danh sách có **chu trình**, `fast` sẽ đuổi kịp `slow` (thuật toán Floyd).

In [5]:
def nut_giua(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    return slow

print("giữa [1..5]:", nut_giua(tao_danh_sach([1, 2, 3, 4, 5])).val,
      "| giữa [1..6]:", nut_giua(tao_danh_sach([1, 2, 3, 4, 5, 6])).val, "(nút giữa thứ hai)")

def co_chu_trinh(head):
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        if slow is fast:
            return True
    return False

v = tao_danh_sach([1, 2, 3, 4])
v.next.next.next.next = v.next          # 4 -> 2 tạo chu trình
print("có chu trình?", co_chu_trinh(v), "| danh sách thường:", co_chu_trinh(tao_danh_sach([1, 2])))

giữa [1..5]: 3 | giữa [1..6]: 4 (nút giữa thứ hai)
có chu trình? True | danh sách thường: False


### Lỗi hay gặp với danh sách liên kết

- Truy cập `cur.next.val` khi `cur.next` là `None` → `AttributeError`. Luôn kiểm tra `cur and cur.next`.
- Đổi `cur.next` **trước** khi lưu nút kế → mất phần còn lại của danh sách.
- Quên trường hợp danh sách rỗng (`head is None`) hoặc chỉ một nút.
- Quên dùng dummy khi `head` có thể bị xóa/thay → phải viết nhánh riêng, dễ sai.
- In danh sách có chu trình bằng vòng `while cur` → lặp vô hạn.

## 2.2 Áp dụng vào bài này

Ghép ba kỹ thuật **danh sách liên kết** ở trên: tìm **giữa** bằng nhanh/chậm, **đảo ngược** nửa sau, rồi **trộn xen kẽ** hai nửa bằng cách đổi con trỏ.

# 3. Nền tảng

In [6]:
from typing import List, Optional
import random


class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next


def build_list(values):
    dummy = tail = ListNode()
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next


def to_list(head, limit=10**6):
    out = []
    while head and len(out) < limit:
        out.append(head.val)
        head = head.next
    return out


def expected_order(values):
    out, i, j = [], 0, len(values) - 1
    while i <= j:
        out.append(values[i])
        if i != j:
            out.append(values[j])
        i += 1; j -= 1
    return out

print(expected_order([1, 2, 3, 4]), expected_order([1, 2, 3, 4, 5]))

[1, 4, 2, 3] [1, 5, 2, 4, 3]


# 4. Baseline: lưu các nút vào mảng

Mảng cho truy cập ngẫu nhiên → dùng hai chỉ số `i` (đầu) và `j` (cuối), nối xen kẽ. `O(n)` bộ nhớ.

In [7]:
def reorder_array(head: Optional[ListNode]) -> None:
    nodes = []
    node = head
    while node:
        nodes.append(node)
        node = node.next
    i, j = 0, len(nodes) - 1
    while i < j:
        nodes[i].next = nodes[j]
        i += 1
        if i == j:
            break
        nodes[j].next = nodes[i]
        j -= 1
    nodes[i].next = None

h = build_list([1, 2, 3, 4, 5]); reorder_array(h); print(to_list(h))

[1, 5, 2, 4, 3]


`nodes[i].next = None` ở cuối rất quan trọng — nút cuối cùng trong thứ tự mới phải kết thúc danh sách, nếu không sẽ tạo vòng.

# 5. Ý tưởng chính: ba bước, `O(1)` bộ nhớ

```text
1 → 2 → 3 → 4 → 5

Bước 1 (tìm giữa):      1 → 2 → 3        4 → 5        (cắt sau slow)
Bước 2 (đảo nửa sau):   1 → 2 → 3        5 → 4
Bước 3 (trộn xen kẽ):   1 → 5 → 2 → 4 → 3
```

**Bước 1:** `slow`, `fast` bắt đầu từ `head`; lặp khi `fast.next and fast.next.next`. Khi dừng, `slow` là **cuối nửa đầu** (nửa đầu dài hơn hoặc bằng nửa sau). Cắt: `second = slow.next; slow.next = None`.

**Bước 2:** đảo `second` bằng `prev / cur / nxt`.

**Bước 3:** lấy lần lượt một nút từ nửa đầu, một nút từ nửa sau.

In [8]:
def reorder(head: Optional[ListNode]) -> None:
    if not head or not head.next:
        return
    slow = fast = head
    while fast.next and fast.next.next:
        slow = slow.next
        fast = fast.next.next
    second = slow.next
    slow.next = None
    prev = None
    while second:
        nxt = second.next
        second.next = prev
        prev = second
        second = nxt
    first, second = head, prev
    while second:
        n1, n2 = first.next, second.next
        first.next = second
        second.next = n1
        first, second = n1, n2

for vals in ([1, 2, 3, 4], [1, 2, 3, 4, 5]):
    h = build_list(vals); reorder(h); print(vals, "->", to_list(h))

[1, 2, 3, 4] -> [1, 4, 2, 3]
[1, 2, 3, 4, 5] -> [1, 5, 2, 4, 3]


# 6. Mô phỏng

In [9]:
def trace_reorder(values):
    head = build_list(values)
    slow = fast = head
    while fast.next and fast.next.next:
        slow = slow.next
        fast = fast.next.next
    second = slow.next
    slow.next = None
    print("Bước 1 — nửa đầu:", to_list(head), "| nửa sau:", to_list(second))
    prev = None
    while second:
        nxt = second.next; second.next = prev; prev = second; second = nxt
    print("Bước 2 — nửa sau đảo:", to_list(prev))
    first, second = head, prev
    print("Bước 3:")
    while second:
        n1, n2 = first.next, second.next
        first.next = second
        second.next = n1
        print(f"   nối {first.val} -> {second.val} -> {n1.val if n1 else None}  | hiện tại: {to_list(head)}")
        first, second = n1, n2
    return to_list(head)

_ = trace_reorder([1, 2, 3, 4, 5, 6])

Bước 1 — nửa đầu: [1, 2, 3] | nửa sau: [4, 5, 6]
Bước 2 — nửa sau đảo: [6, 5, 4]
Bước 3:
   nối 1 -> 6 -> 2  | hiện tại: [1, 6, 2, 3]
   nối 2 -> 5 -> 3  | hiện tại: [1, 6, 2, 5, 3]
   nối 3 -> 4 -> None  | hiện tại: [1, 6, 2, 5, 3, 4]


# 7. Phiên bản đúng format LeetCode

In [10]:
class Solution:
    def reorderList(self, head: Optional[ListNode]) -> None:
        if not head or not head.next:
            return
        slow = fast = head
        while fast.next and fast.next.next:
            slow = slow.next
            fast = fast.next.next
        second = slow.next
        slow.next = None
        prev = None
        while second:
            nxt = second.next
            second.next = prev
            prev = second
            second = nxt
        first, second = head, prev
        while second:
            n1, n2 = first.next, second.next
            first.next = second
            second.next = n1
            first, second = n1, n2

In [11]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<14} PASS  {actual}")


def run(f, values):
    head = build_list(values)
    nodes, node = [], head
    while node:
        nodes.append(node); node = node.next
    assert f(head) is None
    out, node, ids = [], head, []
    while node and len(out) <= len(values):
        out.append(node.val); ids.append(id(node)); node = node.next
    assert len(out) == len(values), "danh sách có vòng hoặc mất nút"
    assert sorted(ids) == sorted(id(n) for n in nodes), "phải dùng lại đúng các nút cũ"
    return out


sol = Solution()
check("Ví dụ 1", run(sol.reorderList, [1, 2, 3, 4]), [1, 4, 2, 3])
check("Ví dụ 2", run(sol.reorderList, [1, 2, 3, 4, 5]), [1, 5, 2, 4, 3])
check("Một nút", run(sol.reorderList, [1]), [1])
check("Hai nút", run(sol.reorderList, [1, 2]), [1, 2])
check("Ba nút", run(sol.reorderList, [1, 2, 3]), [1, 3, 2])

random.seed(143)
for _ in range(3000):
    vals = [random.randint(1, 1000) for _ in range(random.randint(1, 20))]
    exp = expected_order(vals)
    assert run(sol.reorderList, vals) == exp == run(reorder_array, vals)
big = list(range(1, 5 * 10**4 + 1))
check("5·10^4 nút", run(sol.reorderList, big) == expected_order(big), True)
print("3000 danh sách ngẫu nhiên: đúng thứ tự, không vòng, dùng lại nút cũ")

Ví dụ 1        PASS  [1, 4, 2, 3]
Ví dụ 2        PASS  [1, 5, 2, 4, 3]
Một nút        PASS  [1]
Hai nút        PASS  [1, 2]
Ba nút         PASS  [1, 3, 2]


5·10^4 nút     PASS  True
3000 danh sách ngẫu nhiên: đúng thứ tự, không vòng, dùng lại nút cũ


# 8. Vì sao đúng? Độ phức tạp?

- Sau bước 1, nửa đầu có `⌈n/2⌉` nút `L0..Lm`, nửa sau có `⌊n/2⌋` nút `Lm+1..Ln`.
- Sau bước 2, nửa sau là `Ln, Ln−1, …, Lm+1`.
- Bước 3 nối `L0, Ln, L1, Ln−1, …`. Vì nửa sau không dài hơn nửa đầu, vòng dừng khi nửa sau hết; nút còn lại của nửa đầu (nếu `n` lẻ, chính là `Lm`) đã có `next = None` từ lúc cắt.

```text
Time:  O(n)
Space: O(1)
```

# 9. Những lỗi dễ mắc

## Lỗi 1: Quên cắt (`slow.next = None`)

In [12]:
def wrong_no_cut(head):
    slow = fast = head
    while fast.next and fast.next.next:
        slow = slow.next; fast = fast.next.next
    second = slow.next                             # SAI: thiếu slow.next = None
    prev = None
    while second:
        nxt = second.next; second.next = prev; prev = second; second = nxt
    first, second = head, prev
    while second:
        n1, n2 = first.next, second.next
        first.next = second; second.next = n1
        first, second = n1, n2

h = build_list([1, 2, 3, 4])
wrong_no_cut(h)
print("Sai (in tối đa 12 nút):", to_list(h, limit=12))

Sai (in tối đa 12 nút): [1, 4, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3]


Nút cuối nửa đầu (`2`) vẫn trỏ vào `3` (nút đầu nửa sau cũ, giờ là **cuối** nửa sau đã đảo). Khi trộn tới `2`, ta lưu `n1 = 2.next` — chính là `3` — rồi gán `3.next = n1`, tức `3.next = 3`: vòng tự trỏ.

## Lỗi 2: Trộn mà không lưu `next` trước khi nối

In [13]:
def wrong_merge_no_save(head):
    slow = fast = head
    while fast.next and fast.next.next:
        slow = slow.next; fast = fast.next.next
    second = slow.next; slow.next = None
    prev = None
    while second:
        nxt = second.next; second.next = prev; prev = second; second = nxt
    first, second = head, prev
    steps = 0
    while second and steps < 10:
        first.next = second                        # SAI: mất first.next cũ
        second.next = first.next                   # = second -> tự trỏ
        first, second = first.next, second.next
        steps += 1
    return steps

h = build_list([1, 2, 3, 4])
print("Sai : vòng trộn chạy", wrong_merge_no_save(h), "lần (bị chặn ở 10); nút 4 tự trỏ:", h.next.next is h.next)

Sai : vòng trộn chạy 10 lần (bị chặn ở 10); nút 4 tự trỏ: True


Sau `first.next = second`, giá trị cũ của `first.next` đã mất; dòng sau đọc lại `first.next` thì được chính `second` → `second.next = second`. Luôn lưu `n1, n2` **trước** khi sửa con trỏ.

## Lỗi 3: Đổi giá trị thay vì đổi nút

Đề cấm sửa giá trị. (Ví dụ: đọc ra mảng giá trị rồi ghi lại — sai yêu cầu dù kết quả in ra giống.)

## Lỗi 4: Cách mảng nhưng quên `nodes[i].next = None`

Nút kết thúc giữ con trỏ cũ → vòng lặp.

# 10. Mẹo nhận diện pattern

Nhiều bài danh sách liên kết "trung bình" là **ghép các viên gạch**:

| Viên gạch | Bài gốc |
|---|---|
| Tìm giữa (nhanh/chậm) | #876 |
| Đảo ngược | #206, #92 |
| Trộn hai danh sách | #21 |
| Nút giả (dummy) | #2, #86 |

Ví dụ: **#234 Palindrome Linked List** = tìm giữa + đảo nửa sau + so sánh; **#148 Sort List** = tìm giữa + trộn (merge sort); **#2130 Maximum Twin Sum** = tìm giữa + đảo + duyệt song song.

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Áp dụng ba bước cho `1 → 2 → 3 → 4 → 5 → 6 → 7`.

## Bài 2 — Điền chỗ trống

In [14]:
def exercise_2_reorder_list(head: Optional[ListNode]) -> None:
    if not head or not head.next:
        return
    # TODO: bước 1 — tìm giữa và cắt
    # TODO: bước 2 — đảo nửa sau
    # TODO: bước 3 — trộn xen kẽ
    pass


# h = build_list([1, 2, 3, 4]); exercise_2_reorder_list(h); assert to_list(h, 10) == [1, 4, 2, 3]
# h = build_list([1, 2, 3, 4, 5]); exercise_2_reorder_list(h); assert to_list(h, 10) == [1, 5, 2, 4, 3]

## Bài 3 — Biến thể: Maximum Twin Sum of a Linked List (LeetCode #2130)

Danh sách độ dài chẵn `n`; "cặp sinh đôi" là nút `i` và nút `n − 1 − i`. Trả về tổng lớn nhất của một cặp, `O(1)` bộ nhớ.

```python
pair_sum([5, 4, 2, 1])    -> 6
pair_sum([4, 2, 2, 3])    -> 7
```

In [15]:
def exercise_3_pair_sum(head: Optional[ListNode]) -> int:
    # TODO
    pass


# assert exercise_3_pair_sum(build_list([5, 4, 2, 1])) == 6
# assert exercise_3_pair_sum(build_list([4, 2, 2, 3])) == 7
# assert exercise_3_pair_sum(build_list([1, 100000])) == 100001

## Bài 4 — Biến thể: đảo ngược phép reorder

Cho danh sách đã ở dạng `L0 → Ln → L1 → Ln−1 → …`, khôi phục về `L0 → L1 → … → Ln`, tại chỗ, `O(1)` bộ nhớ.

Gợi ý nhanh: tách các nút ở vị trí chẵn và lẻ thành hai danh sách (như #328), đảo danh sách lẻ, rồi nối.

In [16]:
def exercise_4_unreorder(head: Optional[ListNode]) -> None:
    # TODO
    pass


# h = build_list([1, 4, 2, 3]); exercise_4_unreorder(h); assert to_list(h, 10) == [1, 2, 3, 4]
# h = build_list([1, 5, 2, 4, 3]); exercise_4_unreorder(h); assert to_list(h, 10) == [1, 2, 3, 4, 5]
# h = build_list([1]); exercise_4_unreorder(h); assert to_list(h, 10) == [1]

## Bài 5 — Mini challenge: reorder bằng đệ quy

Viết lời giải đệ quy (không đảo danh sách): hàm `go(node, length)` trả về nút "cuối" tương ứng để ghép với `node`. Đệ quy sâu `n/2` nên chỉ dùng cho danh sách ngắn.

Gợi ý nhanh: `go(head, n)` gọi `go(head.next, n − 2)` và nhận lại nút `tail` tương ứng với `head.next`; nút ứng với `head` là `tail.next`.

In [17]:
def exercise_5_reorder_recursive(head: Optional[ListNode]) -> None:
    # TODO
    pass


# h = build_list([1, 2, 3, 4]); exercise_5_reorder_recursive(h); assert to_list(h, 10) == [1, 4, 2, 3]
# h = build_list([1, 2, 3, 4, 5]); exercise_5_reorder_recursive(h); assert to_list(h, 10) == [1, 5, 2, 4, 3]

# 12. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_reorder` sau khi tự làm.

**Bài 3**
- Mức 1: tìm giữa (danh sách chẵn), đảo nửa sau, duyệt song song hai nửa, lấy max tổng.

**Bài 4**
- Mức 1: `odd = head`, `even_head = even = head.next`; tách như #328.
- Mức 2: đảo danh sách `even`, nối vào cuối danh sách `odd`.

**Bài 5**
- Mức 1: trường hợp gốc: `n == 1` → trả `head`, cắt `head.next = None` sau khi lưu; `n == 2` → trả `head.next`, cắt tương tự.
- Mức 2: `tail = go(head.next, n−2)`; `out = tail.next`; `tail.next = out.next`; `out.next = head.next`; `head.next = out`; trả `tail`.

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [18]:
_ = trace_reorder([1, 2, 3, 4, 5, 6, 7])

Bước 1 — nửa đầu: [1, 2, 3, 4] | nửa sau: [5, 6, 7]
Bước 2 — nửa sau đảo: [7, 6, 5]
Bước 3:
   nối 1 -> 7 -> 2  | hiện tại: [1, 7, 2, 3, 4]
   nối 2 -> 6 -> 3  | hiện tại: [1, 7, 2, 6, 3, 4]
   nối 3 -> 5 -> 4  | hiện tại: [1, 7, 2, 6, 3, 5, 4]


## Đáp án Bài 2

In [19]:
def answer_2_reorder_list(head: Optional[ListNode]) -> None:
    reorder(head)

h = build_list([1, 2, 3, 4]); answer_2_reorder_list(h); assert to_list(h, 10) == [1, 4, 2, 3]
h = build_list([1, 2, 3, 4, 5]); answer_2_reorder_list(h); assert to_list(h, 10) == [1, 5, 2, 4, 3]
print("Bài 2 PASS (giống reorder ở mục 5)")

Bài 2 PASS (giống reorder ở mục 5)


## Đáp án Bài 3

In [20]:
def answer_3_pair_sum(head: Optional[ListNode]) -> int:
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    prev = None
    while slow:
        nxt = slow.next; slow.next = prev; prev = slow; slow = nxt
    best, a, b = 0, head, prev
    while b:
        best = max(best, a.val + b.val)
        a, b = a.next, b.next
    return best

assert answer_3_pair_sum(build_list([5, 4, 2, 1])) == 6
assert answer_3_pair_sum(build_list([4, 2, 2, 3])) == 7
assert answer_3_pair_sum(build_list([1, 100000])) == 100001
for _ in range(1000):
    v = [random.randint(1, 50) for _ in range(2 * random.randint(1, 8))]
    assert answer_3_pair_sum(build_list(v)) == max(v[i] + v[len(v) - 1 - i] for i in range(len(v) // 2))
print("Bài 3 PASS")

Bài 3 PASS


Ở đây dùng `while fast and fast.next` (khác bài chính): với độ dài chẵn, `slow` dừng ở **đầu nửa sau** — đúng điểm cần đảo. Chọn điều kiện nào phụ thuộc vào việc bạn muốn `slow` dừng ở đâu.

## Đáp án Bài 4

In [21]:
def answer_4_unreorder(head: Optional[ListNode]) -> None:
    if not head or not head.next:
        return
    odd, even_head = head, head.next
    even = even_head
    while even and even.next:
        odd.next = even.next
        odd = odd.next
        even.next = odd.next
        even = even.next
    prev, cur = None, even_head
    while cur:
        nxt = cur.next; cur.next = prev; prev = cur; cur = nxt
    odd.next = prev

h = build_list([1, 4, 2, 3]); answer_4_unreorder(h); assert to_list(h, 10) == [1, 2, 3, 4]
h = build_list([1, 5, 2, 4, 3]); answer_4_unreorder(h); assert to_list(h, 10) == [1, 2, 3, 4, 5]
h = build_list([1]); answer_4_unreorder(h); assert to_list(h, 10) == [1]
for _ in range(1000):
    v = list(range(random.randint(1, 15)))
    h = build_list(v); reorder(h); answer_4_unreorder(h)
    assert to_list(h, 100) == v
print("Bài 4 PASS (reorder rồi unreorder trả về danh sách ban đầu)")

Bài 4 PASS (reorder rồi unreorder trả về danh sách ban đầu)


## Đáp án Bài 5

In [22]:
def answer_5_reorder_recursive(head: Optional[ListNode]) -> None:
    n, node = 0, head
    while node:
        n += 1; node = node.next
    def go(h, k):
        if k == 1:
            tail = h
            return tail
        if k == 2:
            return h.next
        tail = go(h.next, k - 2)
        out = tail.next
        tail.next = out.next
        out.next = h.next
        h.next = out
        return tail
    if n <= 2:
        return
    tail = go(head, n)
    tail.next = None

for vals in ([1, 2, 3, 4], [1, 2, 3, 4, 5]):
    h = build_list(vals)
    answer_5_reorder_recursive(h)
    print(vals, "->", to_list(h, 20))
h = build_list([1, 2, 3, 4]); answer_5_reorder_recursive(h); assert to_list(h, 10) == [1, 4, 2, 3]
h = build_list([1, 2, 3, 4, 5]); answer_5_reorder_recursive(h); assert to_list(h, 10) == [1, 5, 2, 4, 3]
for _ in range(1000):
    v = list(range(random.randint(1, 15)))
    h = build_list(v); answer_5_reorder_recursive(h)
    assert to_list(h, 100) == expected_order(v)
print("Bài 5 PASS")

[1, 2, 3, 4] -> [1, 4, 2, 3]
[1, 2, 3, 4, 5] -> [1, 5, 2, 4, 3]
Bài 5 PASS


# 14. Tổng kết

> Ba bước: (1) nhanh/chậm tìm cuối nửa đầu và **cắt**; (2) **đảo** nửa sau; (3) **trộn xen kẽ**. `O(n)` thời gian, `O(1)` bộ nhớ, chỉ thay đổi con trỏ.

---

## Checklist trước khi nộp

- [ ] Cắt `slow.next = None` sau khi tìm giữa?
- [ ] Lưu `first.next`, `second.next` trước khi nối lại?
- [ ] Trộn tới khi nửa sau hết?
- [ ] Không sửa giá trị nút?